# UACB-604 · Lecture 2.4 — One genome, five file formats (R)

> **Colab users:** choose **Runtime → Change runtime type → R** before running.

We download the same tiny genome, **phage phiX174** (5,386 bp), in several formats and read each one with base R.
phiX174 was the first DNA genome ever sequenced (Sanger, 1977), and it is still added to every Illumina run as a control.

**You will:**
1. Read FASTA, GenBank, EMBL and XML versions of the same record and find the same facts in each
2. Read a real FASTQ file of phiX reads and decode its qualities
3. Compare file sizes and explain them with arithmetic

In [ ]:
eutils <- "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_001422.1"
fasta   <- readLines(paste0(eutils, "&rettype=fasta"))
genbank <- readLines(paste0(eutils, "&rettype=gb"))
xml     <- readLines(paste0(eutils, "&rettype=gb&retmode=xml"))
embl    <- readLines("https://www.ebi.ac.uk/ena/browser/api/embl/J02482")
cat("lines: FASTA", length(fasta), "| GenBank", length(genbank), "| EMBL", length(embl), "| XML", length(xml), "\n")

## 1. FASTA: just a name and the sequence
One header line starting with `>`, then the sequence wrapped over several lines.

In [ ]:
head(fasta, 3)
seq <- paste(fasta[-1], collapse = "")
cat("length:", nchar(seq), "\n")
table(strsplit(seq, "")[[1]])

## 2. GenBank: a labelled record
Keywords start at the left margin (`LOCUS`, `DEFINITION`, `FEATURES`, `ORIGIN`); the record ends with `//`.

In [ ]:
head(genbank, 6)
features <- grep("^     (gene|CDS) ", genbank, value = TRUE)
cat("\n", length(grep("^     CDS ", genbank)), "CDS features; first three:\n")
head(features[grepl("CDS", features)], 3)

## 3. EMBL: the same information, two-letter line codes
`ID` identification, `AC` accession, `DE` description, `FT` features, `SQ` sequence header. This one is the original ENA record J02482.

In [ ]:
head(embl, 9)
grep("^SQ", embl, value = TRUE)     # EMBL even states the base counts

✏️ **Exercise 1.** The EMBL `SQ` line gives counts of A, C, G and T. Do they match the counts you computed from the FASTA in section 1? (They come from different databases: ENA's J02482 and NCBI's RefSeq NC_001422.)

## 4. XML: built for computers
Every value sits between named tags, so programs can pick out fields without guessing column positions.

In [ ]:
pick <- function(tag) sub(paste0(".*<", tag, ">(.*)</", tag, ">.*"), "\\1", grep(paste0("<", tag, ">"), xml, value = TRUE)[1])
c(length = pick("GBSeq_length"), topology = pick("GBSeq_topology"), definition = pick("GBSeq_definition"))

## 5. FASTQ: reads plus qualities
Real Illumina MiSeq reads of phiX (run SRR41113426), streamed straight from ENA.

In [ ]:
fq_url <- "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR411/026/SRR41113426/SRR41113426_1.fastq.gz"
fq_file <- "SRR41113426_1.fastq.gz"                 # 0.8 MB; retry if the server is busy
for (attempt in 1:5) {
  ok <- tryCatch({ download.file(fq_url, fq_file, mode = "wb", quiet = TRUE); TRUE }, error = function(e) FALSE)
  if (ok) break; Sys.sleep(3)
}
fq <- readLines(gzfile(fq_file), n = 4000)          # first 1,000 reads (4 lines each)
fq[1:4]
quals <- fq[seq(4, length(fq), by = 4)]
Q <- lapply(quals, function(q) utf8ToInt(q) - 33)   # Phred+33 decoding
cat("read length:", nchar(fq[2]), "| first 10 scores of read 1:", head(Q[[1]], 10), "\n")

In [ ]:
maxlen <- max(lengths(Q))
meanQ <- sapply(1:maxlen, function(i) mean(sapply(Q, function(q) if (length(q) >= i) q[i] else NA), na.rm = TRUE))
plot(meanQ, type = "l", lwd = 2, xlab = "position in read", ylab = "mean Phred quality", main = "MiSeq phiX reads", ylim = c(0, 40))
abline(h = 30, lty = 2, col = "red")

✏️ **Exercise 2.** Which character encodes Q30? Q20? Use `intToUtf8(30 + 33)`. Why does the encoding add 33 and not 0? (Hint: look up which ASCII codes below 33 are printable.)

In [ ]:
# your code here

## 6. File-size arithmetic

In [ ]:
sizes <- c(FASTA = sum(nchar(fasta, "bytes") + 1), GenBank = sum(nchar(genbank, "bytes") + 1),
           EMBL = sum(nchar(embl, "bytes") + 1), XML = sum(nchar(xml, "bytes") + 1))
barplot(sizes / 1000, ylab = "kilobytes", main = "phiX174 in four formats")
round(sizes / 1000, 1)
cat("2-bit minimum for the sequence alone:", 5386 * 2 / 8 / 1000, "kB\n")

✏️ **Exercise 3.** A FASTQ record stores the sequence **and** one quality character per base, plus the header. Estimate the size of an uncompressed FASTQ file for 30× coverage of the human genome (3.1 Gb) with 150 bp reads. (Count reads first; assume ~60 bytes of header and the `+` line per read.) Why is it always stored gzip-compressed?

---
**Further reading:** Buffalo, *Bioinformatics Data Skills*, ch. 10 (working with sequence data) · Pevsner, ch. 2.